In [19]:
print("Hello world")

Hello world


In [20]:
import sys

# Check if we are currently running inside Google Colab
if 'google.colab' in sys.modules:
    print("Colab environment detected. Installing dependencies and cloning repository...")
    # Major project dependencies used by this notebook and local modules.
    colab_requirements = [
        'numpy',
        'sentencepiece',
        'datasets',
        'jax[cpu]',
        'matplotlib',
        'ipykernel',
    ]

    !pip install -q {" ".join(colab_requirements)}
    
    # Run the shell command to clone
    !git clone https://github.com/krishnan1159/NMT.git
    
    # Add the cloned repo to the system path
    sys.path.append('/content/NMT')
    
else:
    print("Local environment detected. Skipping git clone.")

Local environment detected. Skipping git clone.


In [21]:
if 'google.colab' not in sys.modules:
    print("auto-reloading in non google colab case")
    %load_ext autoreload
    %autoreload 2
    %reload_ext autoreload

## Local classes
import importlib
from vocab import Tokenizer
from model_embedding import ModelEmbedding
import config
from config import get_config
import rnn

importlib.reload(rnn)
importlib.reload(config)

auto-reloading in non google colab case
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<module 'config' from '/Users/krishnan/Nexus/projects/NMT/config.py'>

In [22]:
from datasets import load_dataset
import jax
import jax.numpy as jp

In [23]:
ds = load_dataset("shiyue/chr_en", "parallel")

In [24]:
train_set = list(ds["train"]["sentence_pair"])
chr_sentences = [sentence['chr'] for sentence in train_set]
eng_sentences = [sentence['en'] for sentence in train_set]

print(f"Lengths : chr_sentences : {len(chr_sentences)}, eng_sentences : {len(eng_sentences)}")

## Generate vocab for both source and target sentence.
cfg = get_config()
chr_vocab = Tokenizer(chr_sentences, "chr", "chr", "20000")
eng_vocab = Tokenizer(eng_sentences, "eng", "eng", "8000")
random_key = jax.random.key(0)
embedding = ModelEmbedding(chr_vocab.get_vocab_size(), eng_vocab.get_vocab_size(), cfg.embed_size, random_key)

Lengths : chr_sentences : 11639, eng_sentences : 11639


In [25]:
print(eng_sentences[0])
a, b = eng_vocab.to_numpy([eng_sentences[1]])
print(b)
print(a.shape)
embedding.target_embed(a).shape

and by him every one that believeth is justified from all things, from which ye could not be justified by the law of Moses.
[29]
(1, 29)


(1, 29, 64)

In [26]:
chr_sentences = chr_sentences[:64]
eng_sentences = eng_sentences[:64]

chr_tokens, chr_lengths = chr_vocab.to_numpy(chr_sentences, add_bos=True, add_eos=True)
eng_tokens, eng_lengths = eng_vocab.to_numpy(eng_sentences, add_bos=True, add_eos=True)
print(f"Target vocab size : {eng_vocab.get_vocab_size()}, epochs = {cfg.num_epochs}, batch_size = {cfg.batch_size}, total_sents = {chr_tokens.shape}")

rnn.train(chr_tokens, chr_lengths, eng_tokens, eng_lengths, embedding, eng_vocab.get_vocab_size(), cfg)

INFO:rnn:JAX devices: [MpsDevice(id=0)]
INFO:rnn:JAX backend: mps
INFO:rnn:local device count: 1
INFO:rnn:device count: 1
INFO:rnn:RNN cumulative params : embedding = 1792000, encoder = 8193, decoder = 520193 and total = 2320386


Target vocab size : 8000, epochs = 100, batch_size = 32, total_sents = (64, 51)


INFO:rnn:Loss at epoch 1/100 is 8.986589431762695. Time taken to train = 3.788117374991998
INFO:rnn:Loss at epoch 11/100 is 5.399602890014648. Time taken to train = 1.5491634160280228
INFO:rnn:Loss at epoch 21/100 is 5.317826747894287. Time taken to train = 1.549402500037104
INFO:rnn:Loss at epoch 31/100 is 5.260823726654053. Time taken to train = 1.6090064581949264
INFO:rnn:Loss at epoch 41/100 is 5.20279598236084. Time taken to train = 1.5265740000177175
INFO:rnn:Loss at epoch 51/100 is 5.146770000457764. Time taken to train = 1.5508266671095043
INFO:rnn:Loss at epoch 61/100 is 5.055418968200684. Time taken to train = 1.8190919999033213
INFO:rnn:Loss at epoch 71/100 is 5.089386463165283. Time taken to train = 1.7688449169509113
INFO:rnn:Loss at epoch 81/100 is 4.998905181884766. Time taken to train = 1.659078333992511
INFO:rnn:Loss at epoch 91/100 is 4.9504008293151855. Time taken to train = 1.7731492910534143
